In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    # Mobile viewport first (sidebar is lg-only; hamburger opens a drawer - verified in AppShell.jsx)
    driver.set_window_size(390, 844)
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")
    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//section[@aria-label='Pharmacist dashboard']")))
    print("Logged in at mobile size.")

    # Real mobile control: hamburger button aria-label="Open menu" (verified line 481)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='Open menu']"))).click()
    time.sleep(1)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//div[contains(@class, 'staff-drawer-panel')]")))
    print("Mobile drawer opened.")

    # Pick a real menu item inside the drawer (labels verified in NAV_SECTIONS)
    drawer = driver.find_element(By.XPATH, "//div[contains(@class, 'staff-drawer-panel')]")
    drawer.find_element(By.XPATH, ".//button[contains(., 'Customers')]").click()
    time.sleep(2)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='Customers']")))
    # Drawer auto-closes on navigation (verified: setMobileOpen(false) on [active])
    assert not driver.find_elements(By.XPATH, "//div[contains(@class, 'staff-drawer-panel')]"), \
        "Drawer did not close after navigation."
    print("PASS: Mobile navigation opened Customers page")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("59_mobile_navigation_FAIL.png")
finally:
    driver.quit()